# Step 1: lock in the RAMP baseline (official code), resumable via Hugging Face

**One-time setup**
1. Create a Hugging Face *write* token at https://huggingface.co/settings/tokens.
2. In Kaggle: *Add-ons → Secrets → Add secret*, name it `HF_TOKEN`, and enable it for this notebook.
3. Set `HF_REPO` below to `matokebryan/aat-checkpoints`. It is created **private** on first push.

**Each session:** Accelerator *GPU T4 x2*, Internet *on*, then **Save Version → Save & Run All**. The run continues in the background. Each run stops cleanly after 11 h and pushes its checkpoint. Run the notebook again and it resumes from the Hub. Runs that are already finished are skipped automatically.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'   # <- edit once
os.environ['TIME_BUDGET_H'] = '11'
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT
!pip install -q pyyaml
!bash scripts/ramp_official.sh setup

## Sanity check: the official pretrained ℓ∞ model (expect ≈ 83.7 / 48.1 / 59.8 / 7.7 / 38.5)

In [ ]:
if not os.path.exists('runs_official/pretr_linf/eval_autoattack.json'):
    !bash scripts/ramp_official.sh pretr 0

## A. Thesis baseline: RAMP from scratch (λ=5, 80 epochs, GP), the setting of thesis Table 7.1 / paper Table 3
Two seeds run in parallel, one per GPU. Re-run the notebook each session until both print `done already`. Then change the seeds to `"2" "3"`, then `"4"`.

In [ ]:
!(bash scripts/ramp_official.sh train ramp_scratch "0" 0 & bash scripts/ramp_official.sh train ramp_scratch "1" 1 & wait)
!grep -h "epoch\]" external/ramp/trained_models/ramp_scratch_l5_s*/log_train.txt | tail -n 4   # epoch time -> sessions needed

In [ ]:
# once the seeds are finished
!bash scripts/ramp_official.sh eval ramp_scratch "0 1" 0
!python scripts/compare_targets.py --runs runs_official --table thesis_table7_1 --map ramp_scratch_official=ramp
!python scripts/compare_targets.py --runs runs_official --table scratch_table3 --map ramp_scratch_official=ramp_l5

## B. Cheap cross-check: RAMP fine-tuning (paper Table 24), 5 seeds, about 1 GPU-hour each. Run it on the GPU time left over in a session

In [ ]:
!(bash scripts/ramp_official.sh train ramp "0 2 4" 0 & bash scripts/ramp_official.sh train ramp "1 3" 1 & wait)
!(bash scripts/ramp_official.sh eval ramp "0 2 4" 0 & bash scripts/ramp_official.sh eval ramp "1 3" 1 & wait)
!python scripts/compare_targets.py --runs runs_official --map ramp_official=ramp_l1.5 pretr_linf=pretr_linf